In [ ]:
import os

# Send diagnostic plots to a temporary folder; the paper renderer handles final assets.
from pathlib import Path as _NotebookPath
import tempfile as _notebook_tempfile
NOTEBOOK_FIGURE_DIR = _NotebookPath(
    os.environ.get("ANNALS_NOTEBOOK_FIGURE_DIR")
    or _notebook_tempfile.mkdtemp(prefix="annals-t_family_logconcave-")
)
NOTEBOOK_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import laplace, t

# Avoid Matplotlib cache warnings on some setups
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 130,
    "axes.grid": True,
})

rng = np.random.default_rng(0)


In [ ]:
# Laplace(0, 1/sqrt(2)): SciPy scale=b gives pdf = exp(-|x|/b)/(2b); Var(X)=2b^2=1.
LAPLACE_SCALE = 1.0 / np.sqrt(2.0)


def laplace_to_student_t_1d(x: np.ndarray, nu: float, eps: float = 1e-12) -> np.ndarray:
    """Monotone 1D pushforward: x~Laplace(0, 1/sqrt(2)) -> y~t(nu) via quantiles.
    """
    u = laplace.cdf(x, scale=LAPLACE_SCALE)
    u = np.clip(u, eps, 1.0 - eps)
    return t.ppf(u, df=nu)


def jacobian_diag_entry_1d(x: np.ndarray, nu: float, eps: float = 1e-12) -> np.ndarray:
    """Derivative d/dx of y = student_t_ppf(F_Laplace(x))."""
    y = laplace_to_student_t_1d(x, nu=nu, eps=eps)
    return laplace.pdf(x, scale=LAPLACE_SCALE) / t.pdf(y, df=nu)


def transport_2d(X: np.ndarray, nu: float, eps: float = 1e-12) -> np.ndarray:
    """2D diagonal transport: applies 1D map to each coordinate.

    X: (...,2)
    returns Y: (...,2)
    """
    X = np.asarray(X)
    assert X.shape[-1] == 2
    Y = np.empty_like(X, dtype=float)
    Y[..., 0] = laplace_to_student_t_1d(X[..., 0], nu=nu, eps=eps)
    Y[..., 1] = laplace_to_student_t_1d(X[..., 1], nu=nu, eps=eps)
    return Y


def jacobian_2d(X: np.ndarray, nu: float, eps: float = 1e-12) -> np.ndarray:
    """Jacobian matrices J(x) for the 2D diagonal transport.

    returns J with shape (..., 2, 2)
    """
    X = np.asarray(X)
    assert X.shape[-1] == 2
    j1 = jacobian_diag_entry_1d(X[..., 0], nu=nu, eps=eps)
    j2 = jacobian_diag_entry_1d(X[..., 1], nu=nu, eps=eps)

    J = np.zeros(X.shape[:-1] + (2, 2), dtype=float)
    J[..., 0, 0] = j1
    J[..., 1, 1] = j2
    return J


def log_abs_det_jacobian_2d(X: np.ndarray, nu: float, eps: float = 1e-12) -> np.ndarray:
    X = np.asarray(X)
    j1 = jacobian_diag_entry_1d(X[..., 0], nu=nu, eps=eps)
    j2 = jacobian_diag_entry_1d(X[..., 1], nu=nu, eps=eps)
    return np.log(np.abs(j1)) + np.log(np.abs(j2))


In [ ]:
# --- Plots: 1D Jacobian entry j_{nu}(x) ---

nus = [2.1, 3.0, 5.0, 10.0, 30.0]

x = np.linspace(-4.0, 4.0, 2001)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)

ax = axes[0]
for a in nus:
    j = jacobian_diag_entry_1d(x, nu=a)
    ax.plot(x, j, label=f"nu={a}")
ax.set_title(r"Jacobian diagonal entry $j_\nu(x)$  (Laplace source)")
ax.set_xlabel("x")
ax.set_ylabel(r"$j_\nu(x)$")
ax.set_yscale("log")
ax.legend(ncol=2, fontsize=8)

ax = axes[1]
for a in nus:
    j = jacobian_diag_entry_1d(x, nu=a)
    ax.plot(x, np.log(j), label=f"nu={a}")
ax.set_title(r"$\log j_\nu(x)$  (Laplace source)")
ax.set_xlabel("x")
ax.set_ylabel(r"$\log j_\nu(x)$")
ax.legend(ncol=2, fontsize=8)

plt.show()


In [ ]:
# --- Plots: 2D heatmaps of log|det J_T(x)| over a grid ---

grid_lim = 4.0
n_grid = 301
x1 = np.linspace(-grid_lim, grid_lim, n_grid)
x2 = np.linspace(-grid_lim, grid_lim, n_grid)
X1, X2 = np.meshgrid(x1, x2, indexing="xy")
Xgrid = np.stack([X1, X2], axis=-1)  # (n_grid, n_grid, 2)

fig, axes = plt.subplots(1, len(nus), figsize=(3.2 * len(nus), 3.1), constrained_layout=True)
if len(nus) == 1:
    axes = [axes]

# precompute global color limits for comparability
vals = []
for a in nus:
    vals.append(log_abs_det_jacobian_2d(Xgrid, nu=a))
allv = np.concatenate([v.ravel() for v in vals])
vmin, vmax = np.quantile(allv, [0.02, 0.98])

for ax, a, Z in zip(axes, nus, vals):
    im = ax.imshow(
        Z,
        origin="lower",
        extent=[-grid_lim, grid_lim, -grid_lim, grid_lim],
        cmap="coolwarm",
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        aspect="equal",
    )
    ax.set_title(rf"$\nu={a}$", fontsize=16)

cbar = fig.colorbar(im, ax=axes, shrink=0.9)
cbar.set_label(r"$\log|\det J_{T}|$", fontsize=16)

plt.show()
fig.savefig(NOTEBOOK_FIGURE_DIR / "logdet_heatmaps_t_logconcave.png", dpi=150, bbox_inches="tight")


In [ ]:
from numpy.polynomial.laguerre import laggauss


def log_jacobian_diag_entry_1d(x: np.ndarray, nu: float, eps: float = 1e-12) -> np.ndarray:
    """Stable log of j_nu(x) = laplace_pdf(x) / target_pdf(target_ppf(F_Laplace(x)))."""
    u = laplace.cdf(x, scale=LAPLACE_SCALE)
    u = np.clip(u, eps, 1.0 - eps)
    y = t.ppf(u, df=nu)
    return laplace.logpdf(x, scale=LAPLACE_SCALE) - t.logpdf(y, df=nu)


def laplace_expectation_gauss_laguerre_1d(fun, n: int = 80) -> float:
    """Approximate E[fun(X)] for X~Laplace(0, 1/sqrt(2)) using n-point Gauss–Laguerre.

    Laplace(0, b) density is exp(-|x|/b)/(2b). By symmetry,
        E[fun(X)] = (1/2) ∫_0^∞ e^{-u} (fun(b u) + fun(-b u)) du.
    Gauss–Laguerre nodes/weights integrate ∫_0^∞ e^{-x} g(x) dx ≈ Σ w_i g(x_i).
    """
    xs, ws = laggauss(n)
    b = LAPLACE_SCALE
    return 0.5 * float(np.sum(ws * (fun(b * xs) + fun(-b * xs))))


def estimate_E_sum_sq_jac_diag_2d(nu: float, n_gl: int = 80) -> float:
    """Estimate E_{X~Laplace(0, 1/sqrt(2))^2}[ j1(X)^2 + j2(X)^2 ] for diagonal map.

    Since coords are iid and Jacobian is diag(j(x1), j(x2)):
      E[j1^2 + j2^2] = 2 * E_{X~Laplace(0, 1/sqrt(2))}[j(X)^2].
    """
    def j2_of_z(z):
        logj = log_jacobian_diag_entry_1d(z, nu=nu)
        # avoid overflow in exp for extreme quadrature nodes
        return np.exp(np.clip(2.0 * logj, -745.0, 709.0))

    Ej2 = laplace_expectation_gauss_laguerre_1d(j2_of_z, n=n_gl)
    return 2.0 * Ej2


In [ ]:
# --- Plot: estimated E_laplace[ sum of squares of Jacobian diagonals ] vs nu ---

# Shape grid (denser near heavy-tailed regime)
nu_grid = np.concatenate([
    np.linspace(2.05, 5.0, 25),
    np.linspace(5.0, 30.0, 26)[1:],
    np.linspace(30.0, 120.0, 19)[1:],
])

n_gl = 100
rhs_est = np.array([estimate_E_sum_sq_jac_diag_2d(a, n_gl=n_gl) for a in nu_grid])

# "LHS" overlay: variance of the 2D iid target (trace of covariance)
# For standard t_nu (scale 1): Var = nu/(nu-2) for nu>2.
lhs_var_target = 2.0 * (nu_grid / (nu_grid - 2.0))


# Move the Poincaré constant onto the lower bound side of the inequality:
# Var(target) <= C * E[|grad T|^2]  <=>  Var(target) / C <= E[|grad T|^2].
C_poincare = 2
scaled_lower_bound = lhs_var_target / C_poincare

fig, ax = plt.subplots(1, 1, figsize=(7.6, 3.8), constrained_layout=True)
ax.plot(
    nu_grid,
    rhs_est,
    marker="o",
    markersize=2.5,
    linewidth=1.2,
    label=r"Jacobian Size",
)
ax.plot(
    nu_grid,
    scaled_lower_bound,
    linewidth=2.0,
    linestyle="-",
    color="#d62728",
    label=rf"Scaled Lower Bound  (Var(target) / {C_poincare:g})",
)

ax.set_xlabel(r"degrees of freedom $\nu$", fontsize=12)
ax.set_ylabel(r"size", fontsize=12)
ax.set_yscale("log")
ax.legend(fontsize=11)

plt.show()
fig.savefig(NOTEBOOK_FIGURE_DIR / "poincare_comparison_t_logconcave.png", dpi=150)
